# Interfacing TopologicPy with Neo4j

In [ ]:
# This cell is not needed if you have pip installed topologicpy
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

## 1. Imports

In [ ]:
from topologicpy.Neo4j import Neo4j
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.CellComplex import CellComplex
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from topologicpy.Helper import Helper
from getpass import getpass

## 2. Check the TopologicPy Version

In [ ]:
print("This tutorial requires topologicpy version 0.9.21 or newer.")
print(Helper.Version())

## 3. Set your renderer
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [ ]:
renderer = "vscode"

## 4. Set the Neo4j credentials (local or remote via Aura DB)

In [ ]:
URL = input("Enter your Ne04j URI: ")
USERNAME = input("Enter your Ne04j User name: ")

## 5. Verify that the connection is working
Make sure you enter your password in the dialog box that will appear above.

In [ ]:
driver = Neo4j.Connect(
    url=URL,
    username=USERNAME,
    password=getpass() # This will display a password in a dialog box above
)
print("Driver:", driver)
records = Neo4j.Query(driver, "RETURN 1 AS n")
print("If everything is ok, the following line should print the number 1")
print(records[0]["n"])  # should print 1

## 6. Create an Example Topologic Graph

In [ ]:
cc = CellComplex.Prism(uSides=3, vSides=3, wSides=3)
cells = Topology.Cells(cc)

for i, c in enumerate(cells):
    if i == 5: # We are choosing one cell to make it special to identify it later.
        label = "Special"
        color = "yellow"
    else:
        label = "Room"
        color = "red"
    d = Dictionary.ByKeysValues(
    ["label", "category", "name", "color"],
    [label, "Space", "Room_"+str(i), color])
    c = Topology.SetDictionary(c, d)

# Create a graph of the cellComplex
g = Graph.ByTopology(cc)

# Get its vertices and edges
verts = Graph.Vertices(g)
edges = Graph.Edges(g)

# Specify what the edges mean to neo4j
for edge in edges:
    d = Dictionary.ByKeysValues(
    ["label", "category", "weight"],
    ["ADJACENT_TO", "Adjacency", 1.0])
    edge = Topology.SetDictionary(edge, d)

## 7. A bit of a diversion. Here we are finding the "neighborhood" of the special cell

* We will do the same through Neo4J. But here we show you how to do it directly in TopologicPy
* k is the number of hops to explore from the selected vertex.


In [ ]:
graph_local = Graph.Neigborhood(g, vertices=None, k=2, key="label", value="Special", searchType="equal")
Topology.Show(graph_local,
              vertexSize=20,
              vertexColorKey="color",
              backgroundColor="white",
              width=500,
              height=500,
              renderer=renderer)

## 8. Write the Topologic graph to Neo4j

In [ ]:
# This will delete any previous graphs
Neo4j.Reset(driver)

# This will write the topologic graph to Neo4j
Neo4j.ByGraph(
    driver=driver,
    graph=g,
    nodeLabelKey="label",
    defaultNodeLabel="Node",
    nodeCategoryKey="category",
    defaultNodeCategory=None,
    relationshipTypeKey="label",
    defaultRelationshipType="CONNECTED_TO",
    relationshipCategoryKey="category",
    defaultRelationshipCategory=None,
    bidirectional=True,
    deleteAll=False
)

## 9. Inspect the result in Neo4j Dekstop or Aura DB Browser

Open Neo4j Browser and run:

`MATCH (n) RETURN n`

Then run:

`MATCH (a)-[r]->(b) RETURN a, r, b`

## 10. Test a few Cypher queries from Python

In [ ]:
records = Neo4j.Query(driver, "MATCH (n) RETURN count(n) AS c")
print("Node count:", records[0]["c"])

records = Neo4j.Query(driver, "MATCH ()-[r]->() RETURN count(r) AS c")
print("Relationship count:", records[0]["c"])

records = Neo4j.Query(driver, """
MATCH (a)-[r]->(b)
RETURN a.name AS from_name, type(r) AS rel_type, b.name AS to_name
""")

for record in records:
    print(record["from_name"], record["rel_type"], record["to_name"])

## 11. Read the Neo4j graph back into TopologicPy

In [ ]:
g2 = Neo4j.ToGraph(driver)
print(g2)
Topology.Show(g2,
              vertexSize=20,
              vertexColorKey="color",
              backgroundColor="white",
              width=500,
              height=500,
              renderer=renderer)

## 11. Inspect the Returned Graph

In [ ]:
vertices = Graph.Vertices(g2)
edges = Graph.Edges(g2)

print("Vertices:", len(vertices))
print("Edges:", len(edges))

for i, v in enumerate(vertices):
    d = Dictionary.PythonDictionary(Topology.Dictionary(v))
    print("Vertex", i, d)

for i, e in enumerate(edges):
    d = Dictionary.PythonDictionary(Topology.Dictionary(e))
    print("Edge", i, d)

## 12. Test ToGraph with a custom Cypher query

This is important because the Graph class was designed so the ToGraph method can read not only the whole database, but also a subgraph returned by a query.

In [ ]:
# This query searches for node with degree = n and returns the sub-graph.
n = 4

g3 = Neo4j.ToGraph(
    driver,
    cypher="""
    MATCH (n)-[:ADJACENT_TO]-(m)
    WITH n, COUNT(DISTINCT m) AS degree
    WHERE degree = 4
    RETURN n AS result
    UNION
    MATCH (n)-[:ADJACENT_TO]-(m)
    WITH n, COUNT(DISTINCT m) AS degree
    WHERE degree = 4
    MATCH p=(n)-[:ADJACENT_TO]-(m)
    RETURN p AS result
    """
)

print("Queried graph:", g3)
vertices = Graph.Vertices(g3)
edges = Graph.Edges(g3)

print("Vertices:", len(vertices))
print("Edges:", len(edges))
Topology.Show(g3,
              vertexSize=20,
              vertexColorKey="color",
              backgroundColor="white",
              width=500,
              height=500,
              renderer=renderer)


## 13. Find the "Special" node (yellow) and return its neighborhood with k=2

In [ ]:
records = Neo4j.Query(
    driver,
    """
    MATCH (n)
    WHERE n.color = $value
    RETURN elementId(n) AS nodeId
    LIMIT 1
    """,
    parameters={"value": "yellow"}
)

if not records:
    print("No matching node found.")
    g_local = None
else:
    node_id = records[0]["nodeId"]

    # ---------------------------------------------------
    # Get k=2 neighborhood
    # ---------------------------------------------------

    g_local = Neo4j.Neighborhood(
        driver,
        nodeId=node_id,
        depth=2
    )

    print("Neighborhood graph created.")
    print("Node ID:", node_id)

In [ ]:
Topology.Show(g_local,
              vertexSize=20,
              vertexColorKey="color",
              backgroundColor="white",
              width=500,
              height=500,
              renderer=renderer)

## 13. Close Driver When Done

In [ ]:
Neo4j.Close(driver)